# 1. Simple Route to Greet the User

In [ ]:
# !pip install fastapi uvicorn nest-asyncio

In [ ]:
from fastapi import FastAPI
import uvicorn
import nest_asyncio

# Apply nest_asyncio to allow uvicorn to run in a Jupyter notebook
nest_asyncio.apply()

app = FastAPI()

@app.get('/')
def greet():
    return {'message': 'Hello, World!'}

if __name__ == '__main__':
    # Run the app in the background for notebook compatibility
    # uvicorn.run(app, host='127.0.0.1', port=8000)
    pass

### Accessing the endpoint via curl (Commandline / Terminal)
# curl http://localhost:8000/

# 2. Route with User Input and Computation
- Let's create a route that takes a number as input, squares it, and displays the result.
- Also `/greet/{name}` route to your FastAPI application, which takes the username from the URL and greets them.

- We use FastAPI's automatic path parameter parsing.
- FastAPI automatically returns JSON responses.

In [ ]:
from fastapi import FastAPI
import uvicorn
import nest_asyncio

nest_asyncio.apply()
app = FastAPI()

# Captures a string name from the URL.
@app.get('/greet/{name}')
def greet_user(name: str):
    return {'message': f'Hello, {name}!'}

# The route /square/{number} captures an integer from the URL.
@app.get('/square/{number}')
def square(number: int):
    result = number ** 2
    return {'number': number, 'squared_result': result}

if __name__ == '__main__':
    # uvicorn.run(app, host='127.0.0.1', port=8000)
    pass
    
# Testing with curl
# Replace {name} with any username you like.
# curl http://localhost:8000/greet/Prashant
               
# curl http://localhost:8000/square/5

**Additional Notes:**
- **URL Encoding:**
  - If the name contains spaces or special characters, ensure it's URL-encoded.
  - For example, for "John Doe":
    ```bash
    curl http://localhost:8000/greet/John%20Doe
    ```
- **Error Handling:**
  - FastAPI automatically handles type validation errors (e.g., passing a string to `/square/abc` will return a 422 Unprocessable Entity error automatically).

# 3. Route Demonstrating the GET Method
Let's create a route that accepts query parameters using the GET method.

- `400` is the HTTP status code for `Bad Request`.
- By raising an `HTTPException`, the server indicates to the client that the request was invalid.

In [ ]:
from fastapi import FastAPI, HTTPException
import math
import uvicorn
import nest_asyncio

nest_asyncio.apply()
app = FastAPI()

@app.get('/sqrt')
def compute_sqrt(num1: float, num2: float):
    # Prevent division by zero
    if num2 == 0:
        raise HTTPException(status_code=400, detail='Division by zero is not allowed.')
    
    # Compute the division of num1 by num2
    division_result = num1 / num2
    
    # Ensure the result is non-negative for square root
    if division_result < 0:
        raise HTTPException(status_code=400, detail='Cannot compute square root of a negative number.')
    
    # Compute the square root
    sqrt_result = math.sqrt(division_result)
    
    # Return the result
    return {
        'division_result': division_result,
        'sqrt_result': sqrt_result
    }

if __name__ == '__main__':
    # uvicorn.run(app, host='127.0.0.1', port=8000)
    pass
    
# Testing with curl
# curl "http://localhost:8000/sqrt?num1=16&num2=4"
# curl "http://localhost:8000/sqrt?num2=4"  # FastAPI will auto-return 422 Missing parameter
# curl "http://localhost:8000/sqrt?num1=16&num2=0"
# curl "http://localhost:8000/sqrt?num1=-16&num2=4"
# curl "http://localhost:8000/sqrt?num1=abc&num2=4" # FastAPI will auto-return 422 Type error

### Accessing via Web Browser:
You can also test the endpoint by navigating to:
- Valid Input: http://localhost:8000/sqrt?num1=16&num2=4
- Missing Parameters: http://localhost:8000/sqrt?num1=16
- Division by Zero: http://localhost:8000/sqrt?num1=16&num2=0
- Negative Result: http://localhost:8000/sqrt?num1=-16&num2=4

**FastAPI Bonus:** You can test all these routes interactively by visiting the auto-generated Swagger UI at http://localhost:8000/docs

# 4. Route Demonstrating the POST Method
We'll create a route that accepts data via POST and displays it.

In [ ]:
from fastapi import FastAPI, Form
from fastapi.responses import HTMLResponse
import uvicorn
import nest_asyncio

nest_asyncio.apply()
app = FastAPI()

@app.get('/submit', response_class=HTMLResponse)
def get_form():
    # Display a simple form
    return '''
        <form method="post" action="/submit">
            <input type="text" name="message" placeholder="Enter a message" />
            <input type="submit" />
        </form>
    '''

@app.post('/submit', response_class=HTMLResponse)
def submit_form(message: str = Form('No message')):
    return f'<h1>You submitted:<hr /> {message}</h1>'

if __name__ == '__main__':
    # uvicorn.run(app, host='127.0.0.1', port=8000)
    pass
    
# Access it here: http://127.0.0.1:8000/submit
# curl -X POST -d "message=your message" http://localhost:8000/submit

In [ ]:
# import requests
# # URL of the FastAPI endpoint
# url = 'http://127.0.0.1:8000/submit'
# # Data to be sent in the POST request
# data = {
#     'message': 'Hello, FastAPI!'
# }
# # Make the POST request
# response = requests.post(url, data=data)
# # Print the response content
# print(response.text)

### 4A: Modify your FastAPI endpoint to accept JSON data
FastAPI makes JSON handling incredibly easy using Pydantic models.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import nest_asyncio

nest_asyncio.apply()
app = FastAPI()

# Define the expected JSON structure
class MessagePayload(BaseModel):
    message: str = 'No message'

@app.post('/submit_json')
def submit_json(payload: MessagePayload):
    return {'response': f'You submitted: {payload.message}'}

if __name__ == '__main__':
    # uvicorn.run(app, host='127.0.0.1', port=8000)
    pass

In [ ]:
# import requests
# url = 'http://127.0.0.1:8000/submit_json'
# json_data = {
#     'message': 'Hello, FastAPI with JSON!'
# }
# response = requests.post(url, json=json_data)
# print(response.json())

# 5. Advanced APP Route using HTML templates and CSS rendering
FastAPI uses Jinja2Templates just like Flask.

In [ ]:
from fastapi import FastAPI, Request, Form
from fastapi.responses import HTMLResponse
from fastapi.templating import Jinja2Templates
from fastapi.staticfiles import StaticFiles
import math
import uvicorn
import nest_asyncio
import os

nest_asyncio.apply()
app = FastAPI()

# Create directories if they don't exist for the notebook example
os.makedirs('templates', exist_ok=True)
os.makedirs('static', exist_ok=True)

# Mount static files and templates
# app.mount('/static', StaticFiles(directory='static'), name='static')
templates = Jinja2Templates(directory='templates')

@app.get('/', response_class=HTMLResponse)
def home(request: Request):
    # Assuming home.html exists in templates folder
    # return templates.TemplateResponse('home.html', {'request': request})
    return "<h1>Home Page (Template placeholder)</h1>"

@app.get('/sqrt_template', response_class=HTMLResponse)
def get_sqrt_form(request: Request):
    # return templates.TemplateResponse('form.html', {'request': request})
    return "<h1>Form Page (Template placeholder)</h1>"

@app.post('/sqrt_template', response_class=HTMLResponse)
def compute_sqrt_template(request: Request, num1: float = Form(...), num2: float = Form(...)):
    try:
        if num2 == 0:
            error = 'Division by zero is not allowed.'
            # return templates.TemplateResponse('form.html', {'request': request, 'error': error})
            return f"<h1>Error: {error}</h1>"
            
        division_result = num1 / num2
        if division_result < 0:
            error = 'Cannot compute square root of a negative number.'
            # return templates.TemplateResponse('form.html', {'request': request, 'error': error})
            return f"<h1>Error: {error}</h1>"
            
        sqrt_result = math.sqrt(division_result)
        # return templates.TemplateResponse('result.html', {'request': request, 'division_result': division_result, 'sqrt_result': sqrt_result})
        return f"<h1>Result: {sqrt_result}</h1>"
        
    except (ValueError, TypeError):
        error = 'Invalid input. Please provide numeric values.'
        # return templates.TemplateResponse('form.html', {'request': request, 'error': error})
        return f"<h1>Error: {error}</h1>"

if __name__ == '__main__':
    # uvicorn.run(app, host='127.0.0.1', port=8000)
    pass

# http://127.0.0.1:8000

- app.py: The main FastAPI application script.
- static/: Contains static files like CSS, JavaScript, images, etc. The styles.css file is placed here.
- templates/: Contains HTML templates rendered by Jinja2Templates.
- form.html is the template for the form page.
- result.html is the template for displaying the computation result.